# 21 Subsurface data

<!-- navigation -->
[<kbd>← 20 Geohydrological scheme</kbd>](20%20Geohydrological%20scheme.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>51 Prepare model inputs →</kbd>](51%20Prepare%20model%20inputs.ipynb)

**Optional**: support for notebook 20

Downloads subsurface data along each transect (BRO CPTs and boreholes, REGIS and GeoTOP) as support for the geohydrological scheme.

| | |
|---|---|
| **Input** | rectangle and cross-section line per transect in the transects metadata |
| **Output** | files and figures in the transect folder 'subsurface': BRO CPTs and boreholes (`bro`), REGIS and GeoTOP (`regis_geotop`) |

Downloads subsurface data that help to fill in the geohydrological scheme: the locations, graphs and logs of BRO cone penetration tests (CPT) and boreholes in the rectangle of each transect (with brodata), and REGIS and GeoTOP along the cross-section line (with nlmod), including the REGIS layers in the middle of the cross-section as a table. Downloading can take several minutes per transect.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import pandas as pd
print(f"timflow_levee_toolkit {tlt.__version__}")

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]
download_bro = True              # CPTs and boreholes from the BRO
download_regis_geotop = True     # REGIS and GeoTOP along the cross-section line
zlim = None                      # (zmin, zmax) [m NAP] of the profiles; None = derived from the BRO data

## 1. Download and plot per transect

In [ ]:
meta_by_transect = project.metadata().set_index("transect")
regis_profiles = {}
for transect in transects:
    m = meta_by_transect.loc[transect]
    extent = [m.get(c) for c in ("xmin_extent", "xmax_extent", "ymin_extent", "ymax_extent")]
    has_extent = not any(pd.isna(v) for v in extent)
    line = project.crosssection(transect)  # None if the metadata have no cross-section line
    folder = project.transect_dir(transect, "subsurface")
    title = f"{project.name} - {transect}"
    z_limits = zlim or (-30.0, 5.0)

    if download_bro and has_extent:
        print(f"=== {transect}: BRO")
        bro_folder = folder / "bro"
        bro_folder.mkdir(parents=True, exist_ok=True)
        locations = tlt.get_bro_locations(extent)
        tlt.save_bro_locations(locations, bro_folder / "bro_locations.gpkg")
        f, ax = tlt.plot_bro_locations(locations, extent, line=line, title=transect)
        tlt.finalize_and_save(f, bro_folder, title, tag="bro_locations", file_label=transect)
        if zlim is None:
            z_limits = tlt.suggest_z_limits(locations) or z_limits
        if locations["cpt"] is not None and len(locations["cpt"]) > 0:
            tlt.download_cpt_graphs(locations["cpt"], bro_folder)
        if locations["bhr"] is not None and len(locations["bhr"]) > 0:
            tlt.download_bhr_graphs(locations["bhr"], bro_folder, z_limits)
        if line is not None and locations["gbo"] is not None and len(locations["gbo"]) > 0:
            ax = tlt.plot_dino_lithology(locations["gbo"], line)
            tlt.finalize_and_save(ax.figure, bro_folder, title, tag="dino_lithology", file_label=transect)

    if download_regis_geotop and line is not None:
        print(f"=== {transect}: REGIS and GeoTOP")
        rg_folder = folder / "regis_geotop"
        rg_folder.mkdir(parents=True, exist_ok=True)
        zmin, zmax = z_limits
        data = tlt.download_regis_geotop(tlt.extent_around_line(line))
        f, axes = tlt.plot_layer_models(data["regis"], line, ds2=data["combined"], zmin=zmin, zmax=zmax,
                                        title1="REGIS", title2="REGIS + GeoTOP")
        tlt.finalize_and_save(f, rg_folder, title, tag="layer_models", file_label=transect)
        f, ax = tlt.plot_geotop_lithology(data["geotop"], line, zmin=zmin, zmax=zmax)
        tlt.finalize_and_save(f, rg_folder, title, tag="geotop_lithology", file_label=transect)
        for variable in ("kh", "kv"):
            f, ax = tlt.plot_geotop_conductivity(data["geotop"], line, variable=variable, zmin=zmin, zmax=zmax)
            tlt.finalize_and_save(f, rg_folder, title, tag=f"geotop_{variable}", file_label=transect)

        # REGIS layers in the middle of the cross-section, as support for the geohydrological scheme
        x_mid, y_mid = (line[0][0] + line[1][0]) / 2, (line[0][1] + line[1][1]) / 2
        regis_profiles[transect] = tlt.regis_profile(data["regis"], x_mid, y_mid)
        tlt.export_to_excel(regis_profiles[transect], rg_folder / "regis_profile.xlsx")

## 2. REGIS layers in the middle of each cross-section

In [ ]:
for transect, profile in regis_profiles.items():
    print(f"=== {transect}")
    display(profile.style.format(precision=2))

---

<!-- navigation -->
[<kbd>← 20 Geohydrological scheme</kbd>](20%20Geohydrological%20scheme.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>51 Prepare model inputs →</kbd>](51%20Prepare%20model%20inputs.ipynb)